# 02 · pandas & the manifest

Pairs with `docs/08-data-layer-lab.md` (Parts C-D). Goal: the DataFrame and grouping
concepts behind `read_manifest` and the chip-grouped split.

In [ ]:
import json, numpy as np, pandas as pd
from collections import Counter

## 1. A DataFrame from records

Our JSON manifest is a list of `{id, species}`. Build a DataFrame from it and
normalize the `id` column name to `chip_id`.

In [ ]:
records = json.loads(open("../configs/manifest.example.json").read())
df = pd.DataFrame(records).rename(columns={"id": "chip_id"})
print(df.shape)
df.head()

In [ ]:
# Your turn: reimplement gpulab.data.dataset.read_manifest's JSON branch, then diff:
from gpulab.data.dataset import read_manifest
theirs = read_manifest("../configs/manifest.example.json")
# assert your DataFrame equals `theirs` (same columns, same rows)
# TODO

## 2. Iterating rows: `itertuples` vs `iterrows`

`itertuples` is fast and gives named, typed fields; `iterrows` boxes each row into a
Series and is slow. `build_dataset` uses `itertuples`.

In [ ]:
for row in df.head(3).itertuples(index=False):
    print(row.chip_id, "->", row.species)

In [ ]:
# Your turn: time itertuples vs iterrows over df with %timeit (or time.perf_counter).
# TODO

## 3. `groupby` + `unique` — the chip-grouped split

We must split **by chip**, never by well, or correlated wells leak across
train/test. Group by species, take the unique chips, and assign whole chips.

In [ ]:
print("species counts:")
print(Counter(df["species"]))

rng = np.random.default_rng(0)
for species, sub in df.groupby("species"):
    chips = sub["chip_id"].unique()
    rng.shuffle(chips)
    n_test = max(1, round(len(chips) * 0.25)) if len(chips) > 1 else 0
    print(f"{species:16s} {len(chips)} chips -> {n_test} to test")

In [ ]:
# Your turn: build train_chips / test_chips sets from the loop above and assert
# they don't overlap (set(train) & set(test) == empty).
# TODO

> **Concepts to note** (copy into your own theory notebook):
> - `groupby` yields `(key, subframe)` pairs.
> - Split by GROUP (chip), not by row (well), to avoid leakage.
> - `default_rng(seed)` gives reproducible shuffles.
> - Single-chip classes: after a grouped split one side may get 0 chips -> noisy
>   per-class metrics. Note which of your 14 species are chip-poor.

## 4. Labels: integer vs one-hot

XGBoost and PyTorch `CrossEntropyLoss` want **integer** labels; one-hot is a
different encoding used by some Keras models. Build both.

In [ ]:
classes = sorted(df["species"].unique())
lut = {c: i for i, c in enumerate(classes)}
y = df["species"].map(lut).to_numpy()
onehot = np.eye(len(classes))[y]
print("classes:", classes)
print("y[:5]   :", y[:5])
print("onehot  :", onehot.shape)

> **Concepts to note** (copy into your own theory notebook):
> - `CrossEntropyLoss` takes integer class indices, NOT one-hot (common trap).
> - `sorted(unique)` makes the class order deterministic across runs.